# Module 6.4 — Multi-Task Hands-On

**Module 6: Vision + Deployment | Week 2 | Lab (Session 6.4)**

*Source: New content (Session 30, 2026-05-18). Pairs with 6.3 — same architecture, your hands on the knobs. Pattern: M05-lab3 fill-in. Session 35: SHARED_6 replaces OVERLAP_5; `is_young` dual-channel mechanism; all-dogs training; Task 6 rewritten.*

---

In 6.3 you saw the multi-task architecture lift adoption AUC by about +0.032 over M06-1's flat-concat baseline — with `is_young` working through two channels at once (tabular input and aux target). Today you run the same experiments yourself, then test the boundaries: where does the lift come from, which aux targets work, what happens if you stack two.

By the end of this lab you'll have a working feel for:

- Reproducing M06-1's flat-concat baseline (sanity check — your numbers should match the lecture)
- Reproducing the +0.032 AUC lift with `is_young` aux at α=2.0
- Where the α sweet spot actually lives on *your* random seed (the lecture said 2.0; verify)
- Whether aux choice matters: `is_mixed_breed` vs `is_young` vs random labels
- (Stretch) Whether two aux heads stack — does is_young + is_mixed_breed beat single-aux?
- (Stretch) What happens when you remove `is_young` from *both* channels

You won't write the model class — it's given (same `MultiTaskMLP` from 6.3). You'll write the training calls and reason about what you see.

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

## Setup — load data and bundles

Same two bundles as 6.1/6.2/6.3 — `petfinder-mini` + `m06-photos`. If you ran 6.3 in this session, the downloads are no-ops.

In [ ]:
PETFINDER_URL  = 'CONFIGURE_ME'  # ← your instructor will give you the M05 bundle URL
M06_PHOTOS_URL = 'CONFIGURE_ME'  # ← your instructor will give you the M06 bundle URL
DATA_DIR = '/content/petfinder'
os.makedirs(DATA_DIR, exist_ok=True)

def fetch_bundle(url, marker_file, label):
    if os.path.exists(f'{DATA_DIR}/{marker_file}'):
        return
    if url == 'CONFIGURE_ME':
        raise RuntimeError(f'Set {label} above to your instructor URL.')
    import gdown
    zip_path = f'{DATA_DIR}/_bundle.zip'
    gdown.download(url=url, output=zip_path, quiet=False, fuzzy=True)
    if not os.path.exists(zip_path):
        raise RuntimeError(f'{label} download failed — check URL and Drive share settings.')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)
    if not os.path.exists(f'{DATA_DIR}/{marker_file}'):
        raise RuntimeError(f'{marker_file} not found after extraction — wrong zip?')

fetch_bundle(PETFINDER_URL,  'train.csv',                   'PETFINDER_URL')
fetch_bundle(M06_PHOTOS_URL, 'petfinder_cnn_features.csv', 'M06_PHOTOS_URL')

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Bundles ready in {DATA_DIR}; device={device}')


In [ ]:
# Data + aux targets + scaled feature arrays — same prep as 6.3.
df = pd.read_csv(f'{DATA_DIR}/train.csv')
dogs = df[df['Type'] == 1].copy().reset_index(drop=True)

IMAGE = [f'img_{i}' for i in range(768)]
photo_df = pd.read_csv(f'{DATA_DIR}/petfinder_cnn_features.csv').set_index('PetID')

# Left join: include zero-photo dogs (zero-padded CNN).
dogs = dogs.merge(photo_df, left_on='PetID', right_index=True, how='left')
has_photo = dogs[IMAGE[0]].notna().values
dogs[IMAGE] = dogs[IMAGE].fillna(0.0)
dogs = dogs.reset_index(drop=True)

y_adopt     = (dogs['AdoptionSpeed'] <= 2).astype(int).values
y_aux_young = (dogs['Age'] < 6).astype(int).values
y_aux_mixed = ((dogs['Breed1'] == 307) | (dogs['Breed2'] != 0)).astype(int).values

# SHARED_6 tabular — is_young is both here AND the aux target (dual-channel).
X_tab = pd.DataFrame({
    'age_adult':      (dogs['Age'] >= 12).astype(int),
    'is_young':       (dogs['Age'] < 6).astype(int),
    'is_mixed_breed': ((dogs['Breed1'] == 307) | (dogs['Breed2'] != 0)).astype(int),
    'gender_male':    (dogs['Gender'] == 1).astype(int),
    'gender_female':  (dogs['Gender'] == 2).astype(int),
    'sterilized_yes': (dogs['Sterilized'] == 1).astype(int),
}).values.astype(np.float32)
X_img = dogs[IMAGE].values.astype(np.float32)

tr, te = train_test_split(
    np.arange(len(y_adopt)), test_size=0.25, random_state=42, stratify=y_adopt,
)
te_eval = te[has_photo[te]]
y_tr, y_te = y_adopt[tr], y_adopt[te_eval]

def scale(X):
    s = StandardScaler().fit(X[tr])
    return s.transform(X[tr]).astype(np.float32), s.transform(X[te_eval]).astype(np.float32)

X_tab_tr, X_tab_te = scale(X_tab)
X_img_tr, X_img_te = scale(X_img)
X_concat_tr = np.concatenate([X_tab_tr, X_img_tr], axis=1)
X_concat_te = np.concatenate([X_tab_te, X_img_te], axis=1)

print(f'Dogs: {len(dogs):,} ({has_photo.sum():,} with photos)  |  Train: {len(tr):,}  |  Test: {len(te_eval):,}')
print(f'Aux target base rates:  is_young {y_aux_young.mean():.3f}  is_mixed_breed {y_aux_mixed.mean():.3f}')
print(f'Tabular features: {X_tab.shape[1]} (SHARED_6)')


## Given — model classes and training helpers

Same `FlatMLP` baseline and `MultiTaskMLP` architecture from 6.3, plus a `train_baseline` and `train_multitask` helper. **Don't modify these.** Your work is in the task cells below — calling these helpers with different aux targets and α values, then reading the results.

In [ ]:
class FlatMLP(nn.Module):
    """M06-1's baseline: one MLP on concat(tab_6, img_768)."""

    def __init__(self, n_in, hidden=(64, 32), dropout=0.3):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MultiTaskMLP(nn.Module):
    """Shared CNN trunk + aux head + concat-with-tab adopt head.

    For Task 5 stretch this supports a list of aux targets — pass `n_aux` > 1
    at construction and supply a stacked y_aux of shape (N, n_aux) at train time.
    """

    def __init__(self, n_img, n_tab, n_aux=1, trunk=(64, 32),
                 adopt_head=(16,), dropout=0.3):
        super().__init__()
        layers, prev = [], n_img
        for h in trunk:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.trunk = nn.Sequential(*layers)
        trunk_dim = prev
        self.aux_head = nn.Linear(trunk_dim, n_aux)
        layers, prev = [], trunk_dim + n_tab
        for h in adopt_head:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.adopt_head = nn.Sequential(*layers)

    def forward(self, x_img, x_tab):
        z = self.trunk(x_img)
        aux_logit = self.aux_head(z)  # shape (batch, n_aux)
        adopt_logit = self.adopt_head(torch.cat([z, x_tab], dim=1)).squeeze(-1)
        return adopt_logit, aux_logit


In [ ]:
def to_t(arr):
    return torch.tensor(arr, dtype=torch.float32, device=device)


def train_baseline(seed, epochs=150):
    """Flat concat. Returns (acc, auc)."""
    torch.manual_seed(seed)
    model = FlatMLP(X_concat_tr.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    Xtr_t, ytr_t, Xte_t = to_t(X_concat_tr), to_t(y_tr), to_t(X_concat_te)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t)).cpu().numpy()
    return float(((probs > 0.5) == y_te).mean()), float(roc_auc_score(y_te, probs))


def train_multitask(y_aux, seed, alpha, epochs=150):
    """Multi-task with a single aux head. `y_aux` is shape (N,). Returns (acc, auc) on adopt."""
    torch.manual_seed(seed)
    model = MultiTaskMLP(n_img=X_img_tr.shape[1], n_tab=X_tab_tr.shape[1], n_aux=1).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    Xi_tr, Xt_tr = to_t(X_img_tr), to_t(X_tab_tr)
    ya_tr = to_t(y_aux[tr]).unsqueeze(-1)   # shape (N, 1)
    yad_tr = to_t(y_tr)
    Xi_te, Xt_te = to_t(X_img_te), to_t(X_tab_te)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        adopt_logit, aux_logit = model(Xi_tr, Xt_tr)
        loss = loss_fn(adopt_logit, yad_tr) + alpha * loss_fn(aux_logit, ya_tr)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        adopt_logit, _ = model(Xi_te, Xt_te)
        probs = torch.sigmoid(adopt_logit).cpu().numpy()
    return float(((probs > 0.5) == y_te).mean()), float(roc_auc_score(y_te, probs))


def train_multitask_stack(y_aux_stack, seed, alphas, epochs=150):
    """Multi-task with multiple aux heads. `y_aux_stack` is shape (N, n_aux); `alphas` is list of length n_aux.

    Combined aux loss = sum(alpha_k * BCE_k). Returns (acc, auc) on adopt.
    """
    torch.manual_seed(seed)
    n_aux = y_aux_stack.shape[1]
    model = MultiTaskMLP(n_img=X_img_tr.shape[1], n_tab=X_tab_tr.shape[1], n_aux=n_aux).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    Xi_tr, Xt_tr = to_t(X_img_tr), to_t(X_tab_tr)
    ya_tr = to_t(y_aux_stack[tr])
    yad_tr = to_t(y_tr)
    Xi_te, Xt_te = to_t(X_img_te), to_t(X_tab_te)
    alpha_t = torch.tensor(alphas, dtype=torch.float32, device=device)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        adopt_logit, aux_logit = model(Xi_tr, Xt_tr)
        per_aux = [loss_fn(aux_logit[:, k], ya_tr[:, k]) for k in range(n_aux)]
        aux_loss = sum(a * l for a, l in zip(alphas, per_aux))
        loss = loss_fn(adopt_logit, yad_tr) + aux_loss
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        adopt_logit, _ = model(Xi_te, Xt_te)
        probs = torch.sigmoid(adopt_logit).cpu().numpy()
    return float(((probs > 0.5) == y_te).mean()), float(roc_auc_score(y_te, probs))


SEEDS = [42, 1, 2]   # 3-seed mean for any comparison; matches the lecture's pace.

---

### Task 1: Reproduce the flat-concat baseline

Sanity check. Train `train_baseline` over the 3 seeds, compute the mean accuracy and AUC, save them as `BASE_ACC` and `BASE_AUC`. Your AUC should land near **0.695** (within ±0.005 across the 3 seeds — single-seed runs vary more).

In [ ]:
# Task 1: your code here


---

### Task 2: Multi-task with `is_young` aux at α=2.0

Train `train_multitask(y_aux_young, seed, alpha=2.0)` over the 3 seeds. Compute the mean AUC and the delta vs `BASE_AUC`. Your delta should land near **+0.032**.

Print the per-seed AUCs alongside the mean — that gives you a sense of the seed noise. The headline result is from all-dogs training with SHARED_6; your 3-seed mean has more variance than a 5-seed mean, so landing in [+0.020, +0.040] is normal.

Remember what's happening here: `is_young` is in both the SHARED_6 tabular input AND it's the aux target. The adopt head gets the demographic fact directly; the aux head teaches the trunk to see youth in photos. Two complementary channels.

In [ ]:
# Task 2: your code here


---

### Task 3: α sweep — find the peak

Loop over `alphas = [0.5, 1.0, 2.0, 4.0, 8.0]`. For each α, run `train_multitask(y_aux_young, seed, alpha)` over the 3 seeds and record mean AUC. Plot AUC vs α with a horizontal line at `BASE_AUC` so the lift is visible.

Where's the peak on your run? The pre-investigation found α=2.0; with 3 seeds vs 5 you may see the peak at 1.0 or 4.0 depending on seed noise. **What you should NOT see:** α=8.0 winning. Past about 4.0 the aux task starts taking over the trunk and adoption suffers.

Hint: `pd.DataFrame` of α/AUC/Δ is easier to read than a printed loop.

In [ ]:
# Task 3: your code here


---

### Task 4: Aux choice — does it matter what you predict?

Three aux targets at α=2.0, 3-seed mean each:

- `y_aux_young` (already available)
- `y_aux_mixed` (already available)
- a **random** label vector you generate with `np.random.default_rng(0).integers(0, 2, size=len(y_adopt))`

Build a small comparison table: aux target / base rate / mean AUC / Δ vs baseline.

**What you're testing:** the lecture claimed the lift is signal transfer, not regularization. If that's right, random aux labels should give a Δ close to zero (~+0.003 noise floor in the pre-investigation), while real targets should give a Δ near +0.017. If random aux gives roughly the same lift as `is_young`, the lift is just "more parameters / more loss terms" — i.e., regularization, and the architecture isn't doing anything interesting.

Write down: which interpretation does your table support?

In [ ]:
# Task 4: your code here


---

### Task 5 (stretch): Do two aux heads stack?

Use `train_multitask_stack` to train with **both** `is_young` and `is_mixed_breed` as aux targets at `alphas=[1.0, 1.0]` (so total aux budget is 2.0 — matches the single-aux peak). 3-seed mean.

Naive intuition: if each aux contributes +0.017 independently, two aux heads at half weight each should give ~+0.034. The lecture spoiled the answer: **they don't stack** — combined lift caps at ~+0.017 because the 32-dim trunk saturates on demographic features. Both aux targets ask the trunk to encode the same kind of information.

Verify the no-stack claim on your run. If you see a meaningful gain over single-aux (say, +0.025+), you've found something the pre-investigation missed — worth flagging.

Hint: build `y_aux_stack = np.stack([y_aux_young, y_aux_mixed], axis=1)` (shape `(N, 2)`) and pass to `train_multitask_stack`.

In [ ]:
# Task 5 (stretch): your code here


---

### Task 6 (stretch): Remove `is_young` entirely — measure the dual-channel contribution

All the tasks above used SHARED_6 with `is_young` as both tabular input and aux target. How much does `is_young` contribute through **both channels combined**?

Build a SHARED_5 tabular array by *dropping* `is_young` from your existing 6 features:

```python
X_tab_5 = np.delete(X_tab, 1, axis=1)  # column 1 = is_young
```

Re-scale it, rebuild `X_concat` with the 5-feature tabular, then run `train_baseline` (flat concat, no aux head at all) on the reduced feature set. 3-seed mean.

Compare to:
- Task 1's flat-concat baseline (SHARED_6, 6 features): ~0.695 AUC
- Task 2's SHARED_6 + MultiTask result: ~0.727 AUC

Without `is_young` entirely — no tabular input, no aux target — you should land near the FlatMLP baseline from the original OVERLAP_5 experiments (~0.676 AUC). The full `is_young` contribution through both channels is the gap from this number to Task 2's result: roughly **+0.05 AUC**, nearly the entire difference between the OVERLAP_5 baseline and the SHARED_6 + MultiTask peak.

The lesson: one feature, doing double duty through two architectural channels, accounts for most of the AUC gain in M06's single-dataset story.

In [ ]:
# Task 6 (stretch): your code here


---

## Writeup

In a markdown cell at the bottom, write **5–7 bullets** answering:

- (Task 1) What was your baseline AUC? Within ±0.005 of the lecture's 0.695?
- (Task 2) What was your multi-task lift over baseline at α=2.0 with `is_young` aux? Near the expected +0.032?
- (Task 3) Where did your α sweep peak? How does that compare to the lecture's claim of α=2.0? What does α=8.0 do?
- (Task 4) Did random aux give roughly the same lift as `is_young`, or roughly zero? What does that tell you about whether the architecture is doing real work?
- (Task 5, optional) Did two aux heads stack to +0.064, or cap near +0.032? What does the answer say about the 32-dim trunk?
- (Task 6, optional) What AUC did you get without `is_young` entirely? How does the gap to Task 2's result compare to the gap from Task 1's baseline?
- One sentence: in your own words, what's the **dual-channel mechanism** and why does it produce a bigger lift than either channel alone?

---

## Summary

- Reproduced the M06-1 flat-concat baseline (~0.695 AUC at 3-seed mean, SHARED_6 + CNN).
- Reproduced the +0.032 AUC lift via multi-task with `is_young` aux at α=2.0.
- Found the α sweet spot on your seeds; saw α=8.0 degrade adoption performance.
- Verified the lift is signal transfer (real aux works, random aux doesn't).
- (Stretch) Saw that two demographic aux heads don't stack — the 32-dim trunk saturates.
- (Stretch) Measured the full dual-channel contribution of `is_young` — one feature through two architectural channels accounts for most of M06's AUC gain.

You now have hands-on experience with the multi-task architecture that produces M06's headline result: **SHARED_6 + CNN + MultiTask = AUC 0.727**. **Next**: 6.5 generalizes back to the cross-dataset framing M05-lab4 introduced — same shared trunk, different heads, Taiwan deployment. Same SHARED_6 features.

---

**Next session (Lecture 6.5):** Cross-dataset deployment — the multi-task PF head Taiwan recipe.